# 🔎 Análise de Consenso entre LLMs

## 1) Setup e Configuração

In [26]:
import sys
from loguru import logger
import pandas as pd
from src.api.schemas.experiment import ExperimentRequest

logger.remove()
logger.add(
    sys.stdout,
    format="<green>{time:HH:mm:ss}</green> | <level>{level: <8}</level> | <level>{message}</level>",
    level="INFO"
)

logger.success("✓ Setup completo")

11:53:31 | SUCCESS  | ✓ Setup completo


In [27]:
import json
from pathlib import Path

experiment = "large_experiment"

# Load JSON from file
config_path = Path(f"../experiments/{experiment}.json")

with open(config_path, "r") as f:
    config_dict = json.load(f)

# Instantiate Pydantic model
EXPERIMENT_CONFIG = ExperimentRequest(**config_dict)

### - Modelos e prompt

In [28]:
DEFAULT_MODELS = EXPERIMENT_CONFIG.models

PROMPT_TYPE = EXPERIMENT_CONFIG.prompt_type

### - Configurações de consenso

In [29]:
consensus_cfg = {
    "threshold": 0.8,
    "strategy": "majority_vote",
    "no_consensus_strategy": "flag_for_review",
}

consensus_threshold = consensus_cfg.get("threshold", 0.8)
consensus_strategy = consensus_cfg.get("strategy", "majority_vote")
no_consensus_strategy = consensus_cfg.get(
    "no_consensus_strategy", "flag_for_review"
)

### - Configurações de dataset

In [30]:
dataset_cfg = EXPERIMENT_CONFIG.dataset_config

dataset_split = dataset_cfg.split
combine_splits = dataset_cfg.combine_splits
sample_size = dataset_cfg.sample_size
random_state = dataset_cfg.random_state

### - Configurações de cache

In [31]:
cache_dir = "C:\\Users\\gabri\\Documents\\GitHub\\llm-annotation\\data\\.cache"

### - Resultados

In [32]:
results_dir = "C:\\Users\\gabri\\Documents\\GitHub\\llm-annotation\\data\\results"

In [33]:
from src.utils.get_latest_results_date import get_latest_results_date

dataset_name = "books"  
specific_date = "latest"

if specific_date == "latest":
    specific_date = get_latest_results_date(
        results_dir,
        dataset_name
    )
    
results_dir = Path(results_dir)
results_dataset_path = results_dir.joinpath(dataset_name, specific_date)

## 2) Carregar dados

### - Dataset

In [34]:
from src.utils.data_loader import load_hf_dataset

texts, categories, ground_truth = load_hf_dataset(
    dataset_name=dataset_name, 
    cache_dir=cache_dir,
    dataset_global_config=dataset_cfg
)

logger.info(f"Textos: {len(texts)}")
logger.info(f"Categorias: {categories}")
logger.info(f"Ground truth: {'Sim' if ground_truth else 'Não'}")

11:53:31 | INFO     | Carregando dataset: books
11:53:31 | INFO     | Baixando parquet direto do HuggingFace Hub


Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


data.parquet:   0%|          | 0.00/23.9M [00:00<?, ?B/s]

11:53:34 | INFO     | Arquivo baixado em: C:\Users\gabri\Documents\GitHub\llm-annotation\data\.cache\hf\datasets--waashk--books\snapshots\b0871fb23c6050a6cc9569cc9fc6e17ff0e1b6a6\data.parquet


c:\Users\gabri\Documents\GitHub\llm-annotation\.venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning:

`huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\gabri\Documents\GitHub\llm-annotation\data\.cache\hf\datasets--waashk--books. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development



11:53:35 | INFO     | Dataset carregado: 33594 exemplos
11:53:35 | INFO     | Categorias extraídas automaticamente: [0, 1, 2, 3, 4, 5, 6, 7]
11:53:35 | INFO     | Dataset embaralhado com seed=42
11:53:35 | INFO     | Coluna de texto: text
11:53:35 | INFO     | Ground truth carregado da coluna 'label'
11:53:35 | INFO     | Textos: 33594
11:53:35 | INFO     | Categorias: [0, 1, 2, 3, 4, 5, 6, 7]
11:53:35 | INFO     | Ground truth: Sim


### - Anotações

In [35]:
df_annotations = pd.read_csv(results_dataset_path.joinpath("annotations.csv"))

### - Métricas

In [36]:
df_metrics = pd.read_csv(results_dataset_path.joinpath("model_metrics.csv"))

df_metrics

,model,accuracy,f1_macro,precision_macro,recall_macro,coverage,error_rate,invalid_predictions_rate
0,qwen3-8b,0.718462,0.650853,0.681947,0.646501,0.999434,0.281538,0.000566
1,llama3.1-8b,0.714503,0.643716,0.683085,0.639832,0.999970,0.285497,0.000030
2,deepseek-r1-8b,0.674228,0.614947,0.653455,0.607187,0.999613,0.325772,0.000387


## 3) Calcular Consenso

In [37]:
from src.llm_annotation_system.consensus.consensus_calculator import ConsensusCalculator
from src.llm_annotation_system.consensus.consensus_evaluator import ConsensusEvaluator

# Inicializar calculador
consensus_calculator = ConsensusCalculator(
    consensus_threshold=consensus_threshold,
    default_strategy=consensus_strategy
)

analyzer = ConsensusEvaluator(
    categories=categories, 
    calculator=consensus_calculator, 
    output_dir=results_dataset_path
)

df_with_consensus = analyzer.compute_consensus(df_annotations)

11:53:36 | INFO     | Executando cálculo de consenso interno...
11:53:36 | INFO     | Calculando consenso...
11:53:36 | SUCCESS  | Consenso calculado:
11:53:36 | INFO     |   Alto (≥80%): 24810 (73.9%)
11:53:36 | INFO     |   Médio (60-80%): 8196 (24.4%)
11:53:36 | INFO     |   Baixo (<60%): 588 (1.8%)
11:53:36 | INFO     |   Problemáticos: 588 (1.8%)
11:53:36 | INFO     |   Itens que precisam de revisão: 0 (0.0%)
11:53:36 | SUCCESS  | Cálculo de consenso finalizado.


### - Estatisticas

In [38]:
logger.info("\n📊 Estatísticas de Consenso:")
logger.info(f"  Média: {df_with_consensus['consensus_score'].mean():.2%}")
logger.info(f"  Mediana: {df_with_consensus['consensus_score'].median():.2%}")
logger.info(f"  Desvio padrão: {df_with_consensus['consensus_score'].std():.2%}")

11:53:36 | INFO     | 
📊 Estatísticas de Consenso:
11:53:36 | INFO     |   Média: 90.70%
11:53:36 | INFO     |   Mediana: 100.00%
11:53:36 | INFO     |   Desvio padrão: 16.20%


### - Distribuição por nível

In [39]:
# Distribuição por nível
levels = df_with_consensus['consensus_level'].value_counts()
logger.info("Distribuição por nível:")
for level, count in levels.items():
    logger.info(f"  {level}: {count} ({count/len(df_with_consensus):.1%})")

11:53:36 | INFO     | Distribuição por nível:
11:53:36 | INFO     |   high: 24810 (73.9%)
11:53:36 | INFO     |   medium: 8196 (24.4%)
11:53:36 | INFO     |   low: 588 (1.8%)


### - Visualizar consenso

In [40]:
from src.llm_annotation_system.consensus.consensus_visualizer import ConsensusVisualizer

visualizer = ConsensusVisualizer(output_dir=results_dataset_path)

In [41]:
visualizer.plot_score_and_levels(
    df_with_consensus=df_with_consensus,
    levels=levels
)

✓ Gráfico salvo: score_and_levels.html


## 4) Análise Detalhada de Consenso

### - Gerando Report

In [42]:
# Gerar relatório
report = analyzer.generate_consensus_report(
    df=df_with_consensus
)

logger.success("✓ Relatório gerado")

11:53:36 | INFO     | Gerando relatório completo de consenso...
11:53:37 | INFO     | Fleiss' Kappa: 0.789 (Bom)
11:53:38 | INFO     | Casos problemáticos: 588
11:53:38 | SUCCESS  | Relatório de consenso gerado com sucesso.
11:53:38 | SUCCESS  | ✓ Relatório gerado


### - Pairwise_agreement

In [43]:
logger.info("\n📊 Gerando heatmap de concordância...")
visualizer.plot_agreement_heatmap(
    agreement_df=report['pairwise_agreement'],
    title='Matriz de Concordância entre Modelos',
)

11:53:38 | INFO     | 
📊 Gerando heatmap de concordância...


✓ Heatmap salvo: agreement_heatmap.html


### - Cohens Kappa

In [44]:
logger.info("\n📊 Gerando heatmap de Cohens_Kappa...")
visualizer.plot_kappa_heatmap(
    kappa_df=report['cohens_kappa']
)

11:53:38 | INFO     | 
📊 Gerando heatmap de Cohens_Kappa...


✓ Heatmap salvo: kappa_heatmap.html


### - Casos problemáticos

In [45]:
# Casos problemáticos
problematic = report.get('problematic_cases')
if problematic is not None and len(problematic) > 0:
    logger.warning(f"\n⚠️  {len(problematic)} casos problemáticos identificados")
    display(problematic)
else:
    logger.success("\n✓ Nenhum caso problemático identificado")

11:53:38 | WARNING  | 
⚠️  588 casos problemáticos identificados


,text_id,text,consensus_score,annotations,entropy
0,607139e7296616fef1a9d0fb943297a0,"outcast #7 . ""the road before us"" in light of ...",0.333333,"{4: 1, 7: 1, 1: 1}",1.584963
1,9817ba149727732ac0515fdc36fe8c76,"strategic moves (hardy boys: casefiles, #43) ....",0.333333,"{7: 1, 0: 1, 4: 1}",1.584963
2,074cdc755f5cdb5622cd7c11c37c12c7,ex-patriots (ex-heroes #2) . it's been two yea...,0.333333,"{2: 1, 7: 1, 1: 1}",1.584963
3,9325bfaaabd61a6eebb0ab09167f8754,"fullmetal alchemist, vol. 4 (fullmetal alchemi...",0.333333,"{7: 1, 1: 1, 2: 1}",1.584963
4,9319661e13d0646cb2f82f6b59dd9fd3,"hunter huntsman's story (ever after high, #0.6...",0.333333,"{6: 1, 0: 1, 7: 1}",1.584963
...,...,...,...,...,...
583,b37aa243dbd6740972f79eacdc652766,"parasyte, volume 1 . it all comes down to this...",0.333333,"{4: 1, 1: 1, 2: 1}",1.584963
584,198bd3ba6b4e9fee44c67e6091e088f6,the vanishing witch . the reign of richard ii ...,0.333333,"{3: 1, 2: 1, 4: 1}",1.584963
585,8a476e85d9eeb0257eff1bd991232093,"reality bites (coletti warlords, #4) . bree ne...",0.333333,"{7: 1, 2: 1, 6: 1}",1.584963
586,50fb4a2d12a9785ce4f2299cd4a507cf,"the champion . in the spring of 1193, seventee...",0.333333,"{3: 1, 7: 1, 6: 1}",1.584963


## 5) Validação com Ground Truth

In [46]:
accuracy, cls_report, cm = analyzer.evaluate_ground_truth(
    df_with_consensus=df_with_consensus
)

visualizer.plot_confusion_matrix(
    cm=cm,
    categories=categories
)

11:53:38 | SUCCESS  | 
🎯 Accuracy: 71.34%
11:53:38 | INFO     | 
Classification Report:
{'-1': {'precision': 0.0, 'recall': 0.0, 'f1-score': 0.0, 'support': 0.0}, '0': {'precision': 0.9653145208700764, 'recall': 0.7008109261630389, 'f1-score': 0.8120672601384767, 'support': 4686.0}, '1': {'precision': 0.9588205388917133, 'recall': 0.7644912849614917, 'f1-score': 0.8506991429860171, 'support': 4934.0}, '2': {'precision': 0.5347342827370615, 'recall': 0.7076534130085038, 'f1-score': 0.6091601543179345, 'support': 4351.0}, '3': {'precision': 0.8462998102466793, 'recall': 0.4882855266038975, 'f1-score': 0.6192724243265759, 'support': 4567.0}, '4': {'precision': 0.7064630572265926, 'recall': 0.936988543371522, 'f1-score': 0.8055579984170258, 'support': 4888.0}, '5': {'precision': 0.9247015610651974, 'recall': 0.8213703099510603, 'f1-score': 0.8699784017278618, 'support': 1226.0}, '6': {'precision': 0.549049049049049, 'recall': 0.7412162162162163, 'f1-score': 0.6308223116733755, 'support': 4

c:\Users\gabri\Documents\GitHub\llm-annotation\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning:

Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.

c:\Users\gabri\Documents\GitHub\llm-annotation\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning:

Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.

c:\Users\gabri\Documents\GitHub\llm-annotation\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning:

Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.



✓ Matriz de confusão salva em: C:\Users\gabri\Documents\GitHub\llm-annotation\data\results\books\2026-03-23_14-32-43\graphics\confusion_matrix.html


## 6) Exportar Resultados

In [47]:
import json

# Criar diretório
results_dir = results_dataset_path.joinpath("summary")
results_dir.mkdir(parents=True, exist_ok=True)

# Salvar CSVs
df_with_consensus.to_csv(results_dir / 'dataset_anotado_completo.csv', index=False)
logger.info(f"✓ Salvos: {len(df_with_consensus)} registros")

# Alta confiança
high_conf = df_with_consensus[df_with_consensus['consensus_score'] >= 0.8]
high_conf.to_csv(results_dir / 'alta_confianca.csv', index=False)
logger.info(f"✓ Alta confiança: {len(high_conf)} registros")

# Necessita revisão
low_conf = df_with_consensus[df_with_consensus['consensus_score'] < 0.8]
low_conf.to_csv(results_dir / 'necessita_revisao.csv', index=False)
logger.info(f"✓ Necessita revisão: {len(low_conf)} registros")

# Sumário JSON
summary = {
    'dataset': {
        'name': dataset_name,
        'total_texts': len(texts),
        'categories': categories,
        'has_ground_truth': ground_truth is not None
    },
    'config': {
        'models': DEFAULT_MODELS,
        'prompt_type': PROMPT_TYPE,
        'total_models': len(DEFAULT_MODELS),
        'use_alternative_params': False,
        'num_repetitions': 1,
        'total_annotations': len(texts) * len(DEFAULT_MODELS) * 1
    },
    'results': {
        'consensus_mean': float(df_with_consensus['consensus_score'].mean()),
        'consensus_median': float(df_with_consensus['consensus_score'].median()),
        'high_consensus': int((df_with_consensus['consensus_level'] == 'high').sum()),
        'medium_consensus': int((df_with_consensus['consensus_level'] == 'medium').sum()),
        'low_consensus': int((df_with_consensus['consensus_level'] == 'low').sum()),
    },
    'metrics': {
        'fleiss_kappa': float(report['fleiss_kappa']),
        'fleiss_interpretation': report['fleiss_interpretation']
    }
}

if ground_truth:
    summary['validation'] = {
        'classification_report': cls_report
    }

with open(results_dir / 'sumario_experimento.json', 'w') as f:
    json.dump(summary, f, indent=2)

logger.success("\n✓ Resultados exportados com sucesso!")

11:53:39 | INFO     | ✓ Salvos: 33594 registros
11:53:40 | INFO     | ✓ Alta confiança: 24810 registros
11:53:40 | INFO     | ✓ Necessita revisão: 8784 registros
11:53:40 | SUCCESS  | 
✓ Resultados exportados com sucesso!


## 7) Análise de Confiança das Anotações

### - Configuração

In [ ]:
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

MODELS     = DEFAULT_MODELS
CONF_COLS  = [f"{m}_rep1_conf"  for m in MODELS]
LABEL_COLS = [f"{m}_consensus"  for m in MODELS]
N_BINS     = 10
CONF_THRESH = 0.95
CONF_BINS  = np.linspace(0, 1, N_BINS + 1)

COLORS = {
    "deepseek-r1-8b": "#e45c3a",
    "qwen3-8b":       "#3a8fe4",
    "llama3.1-8b":    "#3ae47c",
}

logger.success("✓ Configuração de confiança pronta")

### - 7.1) Distribuição de Confiança por Modelo

In [ ]:
fig = go.Figure()
for model in MODELS:
    fig.add_trace(go.Histogram(
        x=df_annotations[f"{model}_rep1_conf"],
        name=model,
        opacity=0.7,
        nbinsx=40,
        marker_color=COLORS.get(model),
    ))

fig.update_layout(
    barmode="overlay",
    title="Distribuição de Confiança por Modelo",
    xaxis_title="Confiança",
    yaxis_title="Frequência",
    legend_title="Modelo",
)
fig.show()

logger.info("\n📊 Estatísticas de Confiança por Modelo:")
print(df_annotations[CONF_COLS].describe().round(4).to_string())

### - 7.2) Confiança Média por Classe (Heatmap)

In [ ]:
rows = []
for model in MODELS:
    label_col = f"{model}_consensus"
    conf_col  = f"{model}_rep1_conf"
    grouped = df_annotations.groupby(label_col)[conf_col].mean().reset_index()
    grouped.columns = ["class", "mean_conf"]
    grouped["model"] = model
    rows.append(grouped)

df_conf_class = pd.concat(rows).pivot(index="model", columns="class", values="mean_conf")

fig = go.Figure(go.Heatmap(
    z=df_conf_class.values,
    x=[str(c) for c in df_conf_class.columns],
    y=df_conf_class.index.tolist(),
    colorscale="RdYlGn",
    zmin=0.5, zmax=1.0,
    text=df_conf_class.values.round(3),
    texttemplate="%{text}",
))
fig.update_layout(
    title="Confiança Média por Modelo × Classe",
    xaxis_title="Classe Predita",
    yaxis_title="Modelo",
)
fig.show()

### - 7.3) Diagrama de Calibração (Reliability Diagram)

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=[0, 1], y=[0, 1],
    mode="lines", line=dict(dash="dash", color="gray"),
    name="Calibração perfeita"
))

for model in MODELS:
    conf_col  = f"{model}_rep1_conf"
    label_col = f"{model}_consensus"
    df_cal = df_annotations[[conf_col, label_col, "ground_truth"]].dropna().copy()
    df_cal["bin"] = pd.cut(df_cal[conf_col], bins=CONF_BINS, labels=False, include_lowest=True)
    df_cal["correct"] = (df_cal[label_col] == df_cal["ground_truth"]).astype(int)

    cal = df_cal.groupby("bin").agg(
        mean_conf=(conf_col, "mean"),
        accuracy=("correct", "mean"),
        count=(conf_col, "count"),
    ).reset_index()

    fig.add_trace(go.Scatter(
        x=cal["mean_conf"],
        y=cal["accuracy"],
        mode="lines+markers",
        name=model,
        marker=dict(size=cal["count"] / cal["count"].max() * 20 + 4, color=COLORS.get(model)),
        line=dict(color=COLORS.get(model)),
    ))

fig.update_layout(
    title="Diagrama de Calibração (Reliability Diagram)<br><sup>Tamanho do ponto proporcional ao nº de amostras no bin</sup>",
    xaxis_title="Confiança Média no Bin",
    yaxis_title="Acurácia Real",
    xaxis=dict(range=[0, 1]),
    yaxis=dict(range=[0, 1]),
)
fig.show()

### - 7.4) Alta Confiança + Erro (Casos Críticos)

In [ ]:
logger.info(f"\n🚨 Casos de Alta Confiança + Erro (conf > {CONF_THRESH:.0%}):")
summary_rows = []
for model in MODELS:
    conf_col  = f"{model}_rep1_conf"
    label_col = f"{model}_consensus"
    df_m = df_annotations[[conf_col, label_col, "ground_truth"]].dropna()
    high_conf = df_m[df_m[conf_col] >= CONF_THRESH]
    errors    = high_conf[high_conf[label_col] != high_conf["ground_truth"]]
    pct_of_high = len(errors) / len(high_conf) if len(high_conf) > 0 else 0
    pct_of_all  = len(errors) / len(df_m)
    summary_rows.append({
        "Modelo":             model,
        "Total Alta Conf":    len(high_conf),
        "Erros Alta Conf":    len(errors),
        "% Erro (alta conf)": f"{pct_of_high:.2%}",
        "% do total":         f"{pct_of_all:.2%}",
    })

df_crit = pd.DataFrame(summary_rows)
print(df_crit.to_string(index=False))

fig = go.Figure()
for col, color in [("Total Alta Conf", "#aaaaaa"), ("Erros Alta Conf", "#e45c3a")]:
    fig.add_trace(go.Bar(
        x=df_crit["Modelo"],
        y=df_crit[col],
        name=col,
        marker_color=color,
    ))
fig.update_layout(
    barmode="group",
    title=f"Alta Confiança (>{CONF_THRESH:.0%}) — Total vs Erros por Modelo",
    xaxis_title="Modelo",
    yaxis_title="Nº de Amostras",
)
fig.show()

### - 7.5) Distribuição de Confiança: Correto vs Errado

In [ ]:
fig = make_subplots(rows=1, cols=len(MODELS), subplot_titles=MODELS)

for i, model in enumerate(MODELS, start=1):
    conf_col  = f"{model}_rep1_conf"
    label_col = f"{model}_consensus"
    df_m = df_annotations[[conf_col, label_col, "ground_truth"]].dropna().copy()
    df_m["correct"] = df_m[label_col] == df_m["ground_truth"]

    for correct, color, label in [(True, "#3ae47c", "Correto"), (False, "#e45c3a", "Errado")]:
        vals = df_m[df_m["correct"] == correct][conf_col]
        fig.add_trace(
            go.Box(y=vals, name=label, marker_color=color, showlegend=(i == 1)),
            row=1, col=i,
        )

fig.update_layout(
    title="Distribuição de Confiança: Correto vs Errado por Modelo",
    boxmode="group",
    height=450,
)
fig.show()

### - 7.6) Expected Calibration Error (ECE)

In [ ]:
logger.info("\n📐 Expected Calibration Error (ECE) — menor é melhor:")
ece_rows = []
for model in MODELS:
    conf_col  = f"{model}_rep1_conf"
    label_col = f"{model}_consensus"
    df_m = df_annotations[[conf_col, label_col, "ground_truth"]].dropna().copy()
    df_m["bin"] = pd.cut(df_m[conf_col], bins=CONF_BINS, labels=False, include_lowest=True)
    df_m["correct"] = (df_m[label_col] == df_m["ground_truth"]).astype(int)

    ece = 0.0
    n = len(df_m)
    for b in range(N_BINS):
        mask = df_m["bin"] == b
        if mask.sum() == 0:
            continue
        acc  = df_m.loc[mask, "correct"].mean()
        conf = df_m.loc[mask, conf_col].mean()
        ece += (mask.sum() / n) * abs(acc - conf)

    ece_rows.append({"Modelo": model, "ECE": round(ece, 4)})
    logger.info(f"  {model:20s}: ECE = {ece:.4f}")

df_ece = pd.DataFrame(ece_rows)
fig = px.bar(
    df_ece, x="Modelo", y="ECE",
    title="Expected Calibration Error por Modelo (menor = melhor calibrado)",
    color="ECE",
    color_continuous_scale="RdYlGn_r",
    text="ECE",
)
fig.update_traces(texttemplate="%{text:.4f}", textposition="outside")
fig.show()

## 10) Resumo Final

In [48]:
logger.info("\n" + "="*80)
logger.success("RESUMO DO EXPERIMENTO")
logger.info("="*80)

logger.info(f"\n📊 Dataset: {dataset_name}")
logger.info(f"  Textos: {len(texts)}")
logger.info(f"  Categorias: {len(categories)}")

logger.info(f"\n🤖 Configuração:")
logger.info(f"  Modelos base: {len(DEFAULT_MODELS)}")
logger.info(f"  Total modelos: {len(DEFAULT_MODELS)}")
logger.info(f"  Alternative params: {False}")
logger.info(f"  Repetições: {1}")

logger.info(f"\n📈 Consenso:")
logger.info(f"  Média: {df_with_consensus['consensus_score'].mean():.2%}")
logger.info(f"  Fleiss' Kappa: {report['fleiss_kappa']:.3f} ({report['fleiss_interpretation']})")

if ground_truth:
    logger.info(f"\n🎯 Validação:")
    logger.info(f"  Accuracy: {accuracy:.2%}")

logger.info(f"\n📁 Arquivos gerados em: {results_dir}/")

logger.success("\n✅ Análise completa!")

11:53:40 | INFO     | 
11:53:40 | SUCCESS  | RESUMO DO EXPERIMENTO
11:53:40 | INFO     | ================================================================================
11:53:40 | INFO     | 
📊 Dataset: books
11:53:40 | INFO     |   Textos: 33594
11:53:40 | INFO     |   Categorias: 8
11:53:40 | INFO     | 
🤖 Configuração:
11:53:40 | INFO     |   Modelos base: 3
11:53:40 | INFO     |   Total modelos: 3
11:53:40 | INFO     |   Alternative params: False
11:53:40 | INFO     |   Repetições: 1
11:53:40 | INFO     | 
📈 Consenso:
11:53:40 | INFO     |   Média: 90.70%
11:53:40 | INFO     |   Fleiss' Kappa: 0.789 (Bom)
11:53:40 | INFO     | 
🎯 Validação:
11:53:40 | INFO     |   Accuracy: 71.34%
11:53:40 | INFO     | 
📁 Arquivos gerados em: C:\Users\gabri\Documents\GitHub\llm-annotation\data\results\books\2026-03-23_14-32-43\summary/
11:53:40 | SUCCESS  | 
✅ Análise completa!
